In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics==8.3.48 --quiet

**Training YOLOv8**

In [ ]:
import os
import shutil
import gc
from tqdm.notebook import tqdm

#Original dataset path
original = "/content/drive/MyDrive/thesis/data/asl_alphabet_train/asl_alphabet_train"

#First half of the dataset folder path
tiny = "/content/asl_half_safe"

if os.path.exists(tiny):
    print("Dataset already exists – skipping creation")
else:
    os.makedirs(tiny, exist_ok=True)
    classes = sorted([d for d in os.listdir(original) if os.path.isdir(os.path.join(original, d))])

    print(f"Copying first half of images per class ({len(classes)} classes)")
    for cls in tqdm(classes):
        src_folder = os.path.join(original, cls)
        dst_folder = os.path.join(tiny, cls)
        os.makedirs(dst_folder, exist_ok=True)

        imgs = sorted(os.listdir(src_folder))
        half_count = len(imgs) // 2

        for i, img in enumerate(imgs[:half_count]):
            shutil.copy2(os.path.join(src_folder, img), os.path.join(dst_folder, img))


            if i % 200 == 0:
                gc.collect()

    print(f"Dataset ready at {tiny} – first half copied safely!")



import os
import shutil
import random
from tqdm.notebook import tqdm

flat_folder = "/content/asl_half_safe"
final_folder = "/content/asl_half_ready"

if os.path.exists(final_folder):
    shutil.rmtree(final_folder)

#Creating training and testing folder
os.makedirs(f"{final_folder}/train")
os.makedirs(f"{final_folder}/val")

print("Converting existing dataset to YOLO format:")

classes = sorted(os.listdir(flat_folder))

for cls in tqdm(classes):
    cls_path = os.path.join(flat_folder, cls)
    if not os.path.isdir(cls_path):
        continue

    images = os.listdir(cls_path)
    random.shuffle(images)

    train_imgs = images[:int(0.8 * len(images))]
    val_imgs   = images[int(0.8 * len(images)):]

    #Creating class folders
    os.makedirs(f"{final_folder}/train/{cls}", exist_ok=True)
    os.makedirs(f"{final_folder}/val/{cls}", exist_ok=True)


    for img in train_imgs:
        shutil.move(os.path.join(cls_path, img), f"{final_folder}/train/{cls}")
    for img in val_imgs:
        shutil.move(os.path.join(cls_path, img), f"{final_folder}/val/{cls}")

shutil.rmtree(flat_folder)

print("New dataset location:")
print(final_folder)


dataset_path = "/content/asl_half_ready"

import os
print("Classes found:", len(os.listdir(dataset_path + "/train")))
print("Example: train/A has", len(os.listdir(dataset_path + "/train/A")), "images")
print("Example: val/A has", len(os.listdir(dataset_path + "/val/A")), "images")


!pip install ultralytics==8.3.48 --quiet

from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")


#Training and saving the best model
model.train(
    data="/content/asl_half_ready",
    epochs=5,
    imgsz=224,
    batch=16,
    workers=2,
    cache=False,
    project="/content/drive/MyDrive/thesis/yolov8_asl_results", #Saving the model
    name="nano_first_half_2025",
    exist_ok=True,
    augment=True,
    patience=15,
)

**Testing the saved model on modified large test set**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/test_set_8596
!cp -r "/content/drive/MyDrive/thesis/large_asl_test_set/test_set_8596" "/content/test_set_8596"
!find /content/test_set_8596 -type f \( -iname "*.jpg" -o -iname "*.png" -o -iname "*.jpeg" \) | wc -l

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict
import torch


#Path to the saved model on drive
model_path = "/content/drive/MyDrive/thesis/yolov8_asl_results/nano_first_half_2025/weights/best.pt"
model = YOLO(model_path)
print("YOLO task:", model.task)


#Path to test set which needs to be tested
test_root = Path("/content/test_set_8596")

#Copying from drive to local storage
!rm -rf /content/test_set_8596
!cp -r "/content/drive/MyDrive/thesis/large_asl_test_set/test_set_8596" "/content/test_set_8596"
!find /content/test_set_8596 -type f \( -iname "*.jpg" -o -iname "*.png" -o -iname "*.jpeg" \) | wc -l

from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict
import torch


#Path to the saved trained model
model_path = "/content/drive/MyDrive/thesis/yolov8_asl_results/nano_first_half_2025/weights/best.pt"
model = YOLO(model_path)
print("YOLO task:", model.task)


#Path to the specific test set
test_root = Path("/content/test_set_8596")

correct = 0
total = 0
per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)

class_dirs = [d for d in sorted(test_root.iterdir()) if d.is_dir()]

for cls_dir in class_dirs:
    cls_name = cls_dir.name

    imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")) + list(cls_dir.glob("*.jpeg"))
    if len(imgs) == 0:
        continue

    for r in model.predict(
        source=[str(p) for p in imgs],
        imgsz=224,
        device=0 if torch.cuda.is_available() else "cpu",
        batch=64,
        half=True,
        verbose=False,
        stream=True
    ):
        pred_label = r.names[r.probs.top1]
        true_label = cls_name

        total += 1
        per_class_total[true_label] += 1

        if pred_label == true_label:
            correct += 1
            per_class_correct[true_label] += 1

#Computing overall accuracy
acc = correct / total if total else 0
print(f"\nOverall Accuracy: {acc*100:.2f}% ({correct}/{total})\n")

#Computing per-class accuracy
print("Per-class accuracy:")
for c in sorted(per_class_total.keys()):
    a = per_class_correct[c] / per_class_total[c]
    print(f"{c:8}: {a*100:6.2f}% ({per_class_correct[c]}/{per_class_total[c]})")

**Confusion matrix creation for alphabet level analysis**

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

#Loading model
model_path = "/content/drive/MyDrive/thesis/yolov8_asl_results/nano_first_half_2025/weights/best.pt"
model = YOLO(model_path)

print("YOLO task:", model.task)
print("Using GPU:", torch.cuda.is_available())

#Test path that we need to test
test_root = Path("/content/test_set_8596")

true_class_names = sorted([d.name for d in test_root.iterdir() if d.is_dir()])

y_true = []
y_pred = []

for cls_name in true_class_names:
    cls_dir = test_root / cls_name
    imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")) + list(cls_dir.glob("*.jpeg"))
    if len(imgs) == 0:
        continue

    for r in model.predict(
        source=[str(p) for p in imgs],
        imgsz=224,
        device=0 if torch.cuda.is_available() else "cpu",
        batch=64,
        half=True,
        verbose=False,
        stream=True
    ):
        pred_label = r.names[r.probs.top1]
        y_true.append(cls_name)
        y_pred.append(pred_label)

all_labels = sorted(set(y_true) | set(y_pred))
label_to_idx = {lab: i for i, lab in enumerate(all_labels)}

y_true_idx = [label_to_idx[l] for l in y_true]
y_pred_idx = [label_to_idx[l] for l in y_pred]

cm = confusion_matrix(y_true_idx, y_pred_idx, labels=list(range(len(all_labels))))

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(7, 7))
sns.heatmap(cm_norm, cmap="Blues", xticklabels=all_labels, yticklabels=all_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Normalized) – YOLOv8")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

save_path = "/content/drive/MyDrive/thesis/yolov8_confusion_matrix_bg_color.png"
plt.savefig(save_path, dpi=300)
plt.show()